# Exploratory Data Analysis (EDA) for the dataset

In [ ]:
# Install required packages
!pip install transformers torch rdkit kaggle scikit-learn -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_data = pd.read_csv('data/neurips-open-polymer-prediction-2025/train.csv')
test_data = pd.read_csv('data/neurips-open-polymer-prediction-2025/test.csv') 

print(train_data.head())
print(test_data.head())

In [ ]:
# Plotting target distributions (checking for skewness, outliers, etc.)
TARGETS = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, target in enumerate(TARGETS):
    data = train_data[target].dropna()
    axes[i].hist(data, bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{target} (n={len(data)})')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

# NaN counts in last subplot
nan_counts = train_data[TARGETS].isna().sum()
axes[5].barh(TARGETS, nan_counts.values, color='salmon')
axes[5].set_title('Missing Values per Target')
axes[5].set_xlabel('Count')

plt.tight_layout()
plt.show()

print('\nTarget statistics:')
print(train_data[TARGETS].describe())

---
# ChemBERTa Multi-Task Regression Model

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from functools import reduce
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
TARGETS    = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'
DATA_DIR   = 'data/neurips-open-polymer-prediction-2025'
MAX_LEN    = 512
BATCH_SIZE = 16
EPOCHS     = 30
LR         = 2e-5
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# ── Load & Merge All Training Data (main + supplement) ─────────────────────────
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

# Supplement datasets
# dataset1: SMILES + TC_mean  ->  maps to Tc
# dataset2: SMILES only       ->  no labels, skip
# dataset3: SMILES + Tg
# dataset4: SMILES + FFV
ds1 = pd.read_csv(f'{DATA_DIR}/train_supplement/dataset1.csv').rename(columns={'TC_mean': 'Tc'})
ds3 = pd.read_csv(f'{DATA_DIR}/train_supplement/dataset3.csv')
ds4 = pd.read_csv(f'{DATA_DIR}/train_supplement/dataset4.csv')

sup_dfs = [ds1[['SMILES', 'Tc']], ds3[['SMILES', 'Tg']], ds4[['SMILES', 'FFV']]]
sup = reduce(lambda l, r: pd.merge(l, r, on='SMILES', how='outer'), sup_dfs)

# Add any missing target columns to sup so shapes align
for t in TARGETS:
    if t not in sup.columns:
        sup[t] = float('nan')

all_train = pd.concat(
    [train_df[['SMILES'] + TARGETS], sup[['SMILES'] + TARGETS]],
    ignore_index=True
).drop_duplicates(subset='SMILES', keep='first').reset_index(drop=True)

print(f'Total training samples : {len(all_train)}')
print(f'Labels per target      :\n{all_train[TARGETS].notna().sum()}')

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────────
class PolymerDataset(Dataset):
    """Tokenises SMILES strings and returns targets with NaN kept as-is."""

    def __init__(self, smiles_list, targets, tokenizer, max_len=512):
        self.smiles    = smiles_list
        self.targets   = targets          # ndarray (N, 5), NaN = missing
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        target = torch.tensor(self.targets[idx], dtype=torch.float32)
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'targets':        target
        }

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
class ChemBERTaRegressor(nn.Module):
    """ChemBERTa encoder + mean-pooling + MLP regression head (5 outputs)."""

    def __init__(self, model_name, n_targets=5, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_targets)
        )

    def forward(self, input_ids, attention_mask):
        out   = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Mean-pool over non-padding token embeddings
        emb   = out.last_hidden_state                          # (B, L, H)
        mask  = attention_mask.unsqueeze(-1).float()           # (B, L, 1)
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.head(self.dropout(pooled))                 # (B, 5)

In [ ]:
# ── Loss & Metrics ─────────────────────────────────────────────────────────────
def masked_mse(preds, targets):
    """MSE computed only over non-NaN target positions."""
    mask = ~torch.isnan(targets)
    if mask.sum() == 0:
        return preds.sum() * 0.0          # differentiable zero
    return nn.functional.mse_loss(preds[mask], targets[mask])

def masked_mae_per_target(preds_np, targets_np):
    """Per-target MAE (ignoring NaN), returned as a dict."""
    maes = {}
    for i, t in enumerate(TARGETS):
        mask = ~np.isnan(targets_np[:, i])
        if mask.sum() > 0:
            maes[t] = np.abs(preds_np[mask, i] - targets_np[mask, i]).mean()
        else:
            maes[t] = float('nan')
    return maes

In [ ]:
# ── Tokeniser & Data Loaders ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

smiles_all = all_train['SMILES'].tolist()
y_all      = all_train[TARGETS].values.astype(float)

smiles_tr, smiles_val, y_tr, y_val = train_test_split(
    smiles_all, y_all, test_size=0.1, random_state=42
)

train_ds = PolymerDataset(smiles_tr,  y_tr,  tokenizer, MAX_LEN)
val_ds   = PolymerDataset(smiles_val, y_val, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────────
model     = ChemBERTaRegressor(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float('inf')
best_state    = None

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        tgts  = batch['targets'].to(DEVICE)
        optimizer.zero_grad()
        loss = masked_mse(model(ids, mask), tgts)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            tgts = batch['targets'].to(DEVICE)
            val_loss += masked_mse(model(ids, mask), tgts).item()

    scheduler.step()
    avg_tr  = train_loss / len(train_loader)
    avg_val = val_loss   / len(val_loader)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  |  Train MSE: {avg_tr:.4f}  |  Val MSE: {avg_val:.4f}')

# Restore best checkpoint
model.load_state_dict(best_state)
torch.save(best_state, 'chemberta_base.pt')
print(f'\nBest Val MSE: {best_val_loss:.4f}  |  Checkpoint saved to chemberta_base.pt')

In [ ]:
# ── Validation MAE per Target ──────────────────────────────────────────────────
model.eval()
all_preds_val, all_tgts_val = [], []

with torch.no_grad():
    for batch in val_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        all_preds_val.append(model(ids, mask).cpu().numpy())
        all_tgts_val.append(batch['targets'].numpy())

preds_val = np.vstack(all_preds_val)
tgts_val  = np.vstack(all_tgts_val)

print('Validation MAE per target:')
for t, mae in masked_mae_per_target(preds_val, tgts_val).items():
    print(f'  {t:8s}: {mae:.4f}')

In [ ]:
# ── Generate Test Predictions ──────────────────────────────────────────────────
dummy_targets = np.full((len(test_df), len(TARGETS)), float('nan'))
test_ds       = PolymerDataset(test_df['SMILES'].tolist(), dummy_targets, tokenizer, MAX_LEN)
test_loader   = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.eval()
test_preds = []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        test_preds.append(model(ids, mask).cpu().numpy())

test_preds = np.vstack(test_preds)
print('Test predictions shape:', test_preds.shape)

In [ ]:
# ── Build submission.csv ───────────────────────────────────────────────────────
submission = pd.DataFrame({'id': test_df['id']})
for i, target in enumerate(TARGETS):
    submission[target] = test_preds[:, i]

submission.to_csv('submission.csv', index=False)
print('submission.csv written:')
print(submission.to_string(index=False))

In [ ]:
# ── Submit to Kaggle ───────────────────────────────────────────────────────────
# Requires ~/.kaggle/kaggle.json with your API credentials.
# Download from: https://www.kaggle.com/settings  ->  API  ->  Create New Token
import subprocess
result = subprocess.run(
    ['kaggle', 'competitions', 'submit',
     '-c', 'neurips-open-polymer-prediction-2025',
     '-f', 'submission.csv',
     '-m', 'ChemBERTa base model - multi-task regression with masked MSE'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)